# Gray-Box AIA72 — Retrospective Cycle 25 Future Evaluation

First clean future-period evaluation of the frozen Mondrian v2 operational policy. **No tuning or policy reselection is performed. Supplementary 2026 remains unopened.**

In [1]:

from pathlib import Path
from datetime import datetime, timezone
import hashlib, json
import numpy as np
import pandas as pd

# ============================================================
# Gray-Box AIA72 — Clean Future Evaluation on Retrospective Cycle 25
# ============================================================
#
# This is the first clean future-period evaluation of the frozen Mondrian v2
# operational policy. No tuning or policy reselection is performed here.
#
# Frozen upstream:
# - probability calibration: raw/raw/raw
# - Mondrian v2 class-conditional conformal thresholds
# - q80/q90/q95 uncertainty/applicability gates
# - primary v2 policy: mondrian_v2_q90
# - alarm thresholds frozen on model_validation
#
# This notebook evaluates only retrospective_cycle25.
# supplementary_2026 remains unopened.

MASTER = Path("/home/abmoses2000/graybox_aia72_master_predictions_v2_20261005/graybox_aia72_master_predictions.csv.gz")
MASTER_SHA = "17300093b88b7c61ecadf5eb28acba0d6d023b340d7740807852dd4352b22b46"

SHARP = Path("/home/abmoses2000/sharp72_predictions_frozen_20261002.csv.gz")
SHARP_SHA = "7368c002a978321cdc67221564f3b7901687badc7aba28ab9df615179d1e017b"

MOND = Path("/home/abmoses2000/graybox_aia72_conformal_mondrian_v2_20261005")
MOND_PROTOCOL = MOND / "protocol.json"
MOND_THRESHOLDS = MOND / "mondrian_thresholds.csv"

POLICY_V2 = Path("/home/abmoses2000/graybox_aia72_policy_mondrian_v2_freeze_20261005")
POLICY_PROTOCOL = POLICY_V2 / "protocol.json"
POLICY_FAMILY = POLICY_V2 / "policy_family_v2.json"

CUTS = Path("/home/abmoses2000/graybox_aia72_policy_freeze_v1_20261005/candidate_cutoffs.csv")

ALARM_DIR = Path("/home/abmoses2000/graybox_aia72_alarm_and_primary_policy_freeze_v1_20261005")
ALARM_PROTOCOL = ALARM_DIR / "protocol.json"
ALARM_THRESHOLDS = ALARM_DIR / "alarm_thresholds.csv"

OUT = Path("/home/abmoses2000/graybox_aia72_cycle25_mondrian_v2_eval_20261005")
if OUT.exists():
    raise SystemExit(f"Refusing to overwrite existing output: {OUT}")
OUT.mkdir(parents=True)

ROLE = "retrospective_cycle25"
EXPECTED_ROWS = 35846
PRIMARY_ALPHA = 0.10
SEEDS = [17,29,43]

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(4*1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def require(cond,msg):
    if not cond:
        raise ValueError(msg)

def entropy_binary(p):
    p=np.asarray(p,float)
    q=np.clip(p,1e-12,1-1e-12)
    return -(q*np.log(q)+(1-q)*np.log(1-q))

def mondrian_state(p,q0,q1):
    p=np.asarray(p,float)
    include0=p<=q0
    include1=(1-p)<=q1
    out=np.full(len(p),"unavailable",dtype=object)
    finite=np.isfinite(p)
    out[finite&(~include0)&(~include1)]="empty"
    out[finite&include0&(~include1)]="singleton_no_flare"
    out[finite&(~include0)&include1]="singleton_flare"
    out[finite&include0&include1]="doubleton"
    return out,include0,include1

def binary_metrics(y,pred):
    y=np.asarray(y,int); pred=np.asarray(pred,int)
    tp=int(((y==1)&(pred==1)).sum())
    fp=int(((y==0)&(pred==1)).sum())
    fn=int(((y==1)&(pred==0)).sum())
    tn=int(((y==0)&(pred==0)).sum())
    recall=tp/(tp+fn) if (tp+fn) else np.nan
    spec=tn/(tn+fp) if (tn+fp) else np.nan
    prec=tp/(tp+fp) if (tp+fp) else np.nan
    f1=2*prec*recall/(prec+recall) if np.isfinite(prec) and np.isfinite(recall) and (prec+recall)>0 else np.nan
    tss=recall+spec-1 if np.isfinite(recall) and np.isfinite(spec) else np.nan
    denom=((tp+fn)*(fn+tn)+(tp+fp)*(fp+tn))
    hss=2*(tp*tn-fn*fp)/denom if denom else np.nan
    return dict(tp=tp,fp=fp,fn=fn,tn=tn,recall=float(recall),specificity=float(spec),
                precision=float(prec),f1=float(f1),tss=float(tss),hss=float(hss))

def brier(y,p):
    m=np.isfinite(p)
    return float(np.mean((p[m]-y[m])**2)) if m.any() else None

def logloss(y,p):
    m=np.isfinite(p)
    if not m.any(): return None
    q=np.clip(p[m],1e-12,1-1e-12); yy=y[m]
    return float(-np.mean(yy*np.log(q)+(1-yy)*np.log(1-q)))

# -------- integrity --------
for p in [MASTER,SHARP,MOND_PROTOCOL,MOND_THRESHOLDS,POLICY_PROTOCOL,POLICY_FAMILY,CUTS,ALARM_PROTOCOL,ALARM_THRESHOLDS]:
    require(p.is_file(),f"Missing frozen input: {p}")
require(sha256(MASTER)==MASTER_SHA,"Master SHA changed")
require(sha256(SHARP)==SHARP_SHA,"SHARP SHA changed")

mond_protocol=json.loads(MOND_PROTOCOL.read_text())
policy_protocol=json.loads(POLICY_PROTOCOL.read_text())
policy_family=json.loads(POLICY_FAMILY.read_text())
alarm_protocol=json.loads(ALARM_PROTOCOL.read_text())

require(mond_protocol["policy_validation_used_to_fit_v2"] is False,"v2 Mondrian contaminated")
require(policy_protocol["policy_validation_used_to_fit_or_select_v2"] is False,"v2 policy contaminated")
require(policy_protocol["retrospective_cycle25_outcomes_read"] is False,"Cycle25 already read by freeze")
require(policy_protocol["supplementary_2026_outcomes_read"] is False,"2026 already read by freeze")
require(policy_protocol["primary_policy_id"]=="mondrian_v2_q90","Primary v2 policy changed")

cuts=pd.read_csv(CUTS)
mond=pd.read_csv(MOND_THRESHOLDS)
alarm=pd.read_csv(ALARM_THRESHOLDS)
alarm_map={r.branch:float(r.threshold) for r in alarm.itertuples()}

q={}
for branch in ["sharp","aia","sharp_aia"]:
    hit=mond.loc[(mond["branch"]==branch)&np.isclose(mond["alpha"].astype(float),PRIMARY_ALPHA)]
    require(len(hit)==1,f"Missing qhat for {branch}")
    q[branch]=(float(hit.iloc[0]["qhat_class0"]),float(hit.iloc[0]["qhat_class1"]))

def cutoff(metric,pct):
    hit=cuts.loc[(cuts["metric"]==metric)&np.isclose(cuts["percentile"].astype(float),pct)]
    require(len(hit)==1,f"Missing cutoff {metric} q{pct}")
    return float(hit.iloc[0]["cutoff"])

# -------- open Cycle 25 only --------
master=pd.read_csv(MASTER,low_memory=False)
sharp=pd.read_csv(SHARP,low_memory=False)
a=master.loc[master["role"].astype(str)==ROLE].copy()
s=sharp.loc[sharp["role"].astype(str)==ROLE].copy()
require(len(a)==EXPECTED_ROWS and len(s)==EXPECTED_ROWS,"Cycle25 support changed")

a["forecast_case_id"]=a["forecast_case_id"].astype(str)
s["forecast_case_id"]=s["forecast_case_id"].astype(str)
require(a["forecast_case_id"].tolist()==s["forecast_case_id"].tolist(),"SHARP/AIA Cycle25 ID mismatch")

y=pd.to_numeric(a["label"],errors="raise").astype(int).to_numpy()
ys=pd.to_numeric(s["label"],errors="raise").astype(int).to_numpy()
require(np.array_equal(y,ys),"Cycle25 labels disagree")

aia_seed=a[[f"probability_seed_{z}" for z in SEEDS]].astype(float).to_numpy()
sharp_seed=s[[f"probability_seed_{z}" for z in SEEDS]].astype(float).to_numpy()
aia_p=pd.to_numeric(a["probability"],errors="coerce").to_numpy(float)
sharp_p=pd.to_numeric(s["probability_gru_mean"],errors="coerce").to_numpy(float)
fusion_seed=0.5*sharp_seed+0.5*aia_seed
fusion_p=0.5*sharp_p+0.5*aia_p

aia_status=a["input_status"].astype(str).to_numpy() if "input_status" in a.columns else np.array(["ok"]*len(a),dtype=object)
aia_ok=(aia_status=="ok")&np.isfinite(aia_p)&np.isfinite(aia_seed).all(axis=1)
sharp_ok=np.isfinite(sharp_p)&np.isfinite(sharp_seed).all(axis=1)
fusion_ok=aia_ok&sharp_ok&np.isfinite(fusion_p)

sig=pd.DataFrame({
    "forecast_case_id":a["forecast_case_id"].to_numpy(),
    "issue_utc":a["issue_utc"].to_numpy() if "issue_utc" in a.columns else pd.NA,
    "label":y,
    "aia_input_status":aia_status,
    "sharp_input_status":np.where(sharp_ok,"ok","unavailable"),
    "aia_probability":aia_p,
    "sharp_probability":sharp_p,
    "fusion_probability":fusion_p,
    "aia_seed_std":np.nanstd(aia_seed,axis=1,ddof=1),
    "sharp_seed_std":np.nanstd(sharp_seed,axis=1,ddof=1),
    "fusion_seed_std":np.nanstd(fusion_seed,axis=1,ddof=1),
    "sharp_aia_probability_gap":np.abs(sharp_p-aia_p),
    "fusion_entropy":entropy_binary(fusion_p),
})

for branch,p in [("sharp",sharp_p),("aia",aia_p),("sharp_aia",fusion_p)]:
    st,inc0,inc1=mondrian_state(p,*q[branch])
    sig[f"{branch}_mondrian_state"]=st
    sig[f"{branch}_contains_true"]=np.where(y==0,inc0,inc1)

coverage_rows=[]
for branch in ["sharp","aia","sharp_aia"]:
    state=sig[f"{branch}_mondrian_state"].to_numpy()
    contains=sig[f"{branch}_contains_true"].to_numpy(bool)
    finite=state!="unavailable"
    coverage_rows.append({
        "branch":branch,
        "alpha":PRIMARY_ALPHA,
        "available_cases":int(finite.sum()),
        "overall_coverage":float(contains[finite].mean()),
        "class0_coverage":float(contains[(y==0)&finite].mean()),
        "class1_coverage":float(contains[(y==1)&finite].mean()),
        "empty_rate":float((state[finite]=="empty").mean()),
        "singleton_no_flare_rate":float((state[finite]=="singleton_no_flare").mean()),
        "singleton_flare_rate":float((state[finite]=="singleton_flare").mean()),
        "doubleton_rate":float((state[finite]=="doubleton").mean()),
    })
coverage=pd.DataFrame(coverage_rows)

# Branch baseline performance on all available cases.
branch_rows=[]
for branch,p,available in [("sharp",sharp_p,sharp_ok),("aia",aia_p,aia_ok),("sharp_aia",fusion_p,fusion_ok)]:
    pred=(p>=alarm_map[branch]).astype(int)
    m=binary_metrics(y[available],pred[available])
    branch_rows.append({
        "branch":branch,
        "available_cases":int(available.sum()),
        "availability_rate":float(available.mean()),
        "brier":brier(y,p),
        "log_loss":logloss(y,p),
        **m
    })
branch_metrics=pd.DataFrame(branch_rows)

# Always-issue fusion comparator.
baseline_issue=fusion_ok
baseline_pred=(fusion_p>=alarm_map["sharp_aia"]).astype(int)
baseline_metrics=binary_metrics(y[baseline_issue],baseline_pred[baseline_issue])

def eval_policy(pid,pct):
    fs=cutoff("fusion_seed_std",pct)
    gap=cutoff("sharp_aia_probability_gap",pct)
    ent=cutoff("fusion_entropy",pct)
    ss=cutoff("sharp_seed_std",pct)
    fusion_single=np.isin(sig["sharp_aia_mondrian_state"].to_numpy(),["singleton_no_flare","singleton_flare"])
    sharp_single=np.isin(sig["sharp_mondrian_state"].to_numpy(),["singleton_no_flare","singleton_flare"])
    normal=(fusion_ok&fusion_single&(sig["fusion_seed_std"].to_numpy()<=fs)&
            (sig["sharp_aia_probability_gap"].to_numpy()<=gap)&
            (sig["fusion_entropy"].to_numpy()<=ent))
    degraded=((~normal)&sharp_ok&sharp_single&(sig["sharp_seed_std"].to_numpy()<=ss))
    abstain=~(normal|degraded)
    pred=np.full(len(y),-1,dtype=int)
    pred[normal]=(fusion_p[normal]>=alarm_map["sharp_aia"]).astype(int)
    pred[degraded]=(sharp_p[degraded]>=alarm_map["sharp"]).astype(int)
    issued=normal|degraded
    m=binary_metrics(y[issued],pred[issued])
    positives=int((y==1).sum())
    row={
        "policy_id":pid,
        "cutoff_percentile":pct,
        "requested_cases":len(y),
        "positives":positives,
        "normal_cases":int(normal.sum()),
        "degraded_cases":int(degraded.sum()),
        "abstain_cases":int(abstain.sum()),
        "availability_rate":float(issued.mean()),
        "positive_retention_rate":float(((y==1)&issued).sum()/positives),
        "positive_abstention_rate":float(((y==1)&abstain).sum()/positives),
        "detected_flares":int(((y==1)&issued&(pred==1)).sum()),
        "issued_but_missed_flares":int(((y==1)&issued&(pred==0)).sum()),
        "abstained_flares":int(((y==1)&abstain).sum()),
        "false_alarms":int(((y==0)&issued&(pred==1)).sum()),
        **{f"issued_{k}":v for k,v in m.items()}
    }
    sig[f"{pid}_state"]=np.where(normal,"NORMAL",np.where(degraded,"DEGRADED","ABSTAIN"))
    return row

rows=[]
rows.append({
    "policy_id":"baseline_always_issue_fusion_when_available",
    "cutoff_percentile":np.nan,
    "requested_cases":len(y),
    "positives":int((y==1).sum()),
    "normal_cases":int(baseline_issue.sum()),
    "degraded_cases":0,
    "abstain_cases":int((~baseline_issue).sum()),
    "availability_rate":float(baseline_issue.mean()),
    "positive_retention_rate":float(((y==1)&baseline_issue).sum()/(y==1).sum()),
    "positive_abstention_rate":float(((y==1)&(~baseline_issue)).sum()/(y==1).sum()),
    "detected_flares":int(((y==1)&baseline_issue&(baseline_pred==1)).sum()),
    "issued_but_missed_flares":int(((y==1)&baseline_issue&(baseline_pred==0)).sum()),
    "abstained_flares":int(((y==1)&(~baseline_issue)).sum()),
    "false_alarms":int(((y==0)&baseline_issue&(baseline_pred==1)).sum()),
    **{f"issued_{k}":v for k,v in baseline_metrics.items()}
})
rows.append(eval_policy("mondrian_v2_q80",0.80))
rows.append(eval_policy("mondrian_v2_q90",0.90))
rows.append(eval_policy("mondrian_v2_q95",0.95))
results=pd.DataFrame(rows)

primary=results.loc[results["policy_id"]=="mondrian_v2_q90"].iloc[0]
baseline=results.loc[results["policy_id"]=="baseline_always_issue_fusion_when_available"].iloc[0]

comparison={
    "primary_policy_id":"mondrian_v2_q90",
    "baseline_policy_id":"baseline_always_issue_fusion_when_available",
    "availability_delta":float(primary["availability_rate"]-baseline["availability_rate"]),
    "positive_retention_delta":float(primary["positive_retention_rate"]-baseline["positive_retention_rate"]),
    "issued_tss_delta":float(primary["issued_tss"]-baseline["issued_tss"]),
    "false_alarm_delta":int(primary["false_alarms"]-baseline["false_alarms"]),
    "abstained_flare_delta":int(primary["abstained_flares"]-baseline["abstained_flares"]),
}

results_path=OUT/"cycle25_policy_results.csv"
coverage_path=OUT/"cycle25_mondrian_coverage.csv"
branch_path=OUT/"cycle25_branch_metrics.csv"
cases_path=OUT/"cycle25_policy_cases.csv.gz"
summary_path=OUT/"summary.json"

results.to_csv(results_path,index=False)
coverage.to_csv(coverage_path,index=False)
branch_metrics.to_csv(branch_path,index=False)
sig.to_csv(cases_path,index=False,compression={"method":"gzip","mtime":0})

summary={
    "status":"CYCLE25_MONDRIAN_V2_FUTURE_EVALUATION_COMPLETE",
    "scientific_acceptance":False,
    "created_utc":datetime.now(timezone.utc).isoformat(),
    "role":ROLE,
    "rows":len(y),
    "positives":int((y==1).sum()),
    "primary_policy_id":"mondrian_v2_q90",
    "policy_reselection_performed":False,
    "supplementary_2026_outcomes_read":False,
    "comparison_primary_vs_baseline":comparison,
    "source_sha256":{
        "master_predictions":MASTER_SHA,
        "sharp_predictions":SHARP_SHA,
        "mondrian_protocol":sha256(MOND_PROTOCOL),
        "mondrian_thresholds":sha256(MOND_THRESHOLDS),
        "policy_v2_protocol":sha256(POLICY_PROTOCOL),
        "policy_v2_family":sha256(POLICY_FAMILY),
        "candidate_cutoffs":sha256(CUTS),
        "alarm_protocol":sha256(ALARM_PROTOCOL),
        "alarm_thresholds":sha256(ALARM_THRESHOLDS),
    },
    "output_sha256":{
        results_path.name:sha256(results_path),
        coverage_path.name:sha256(coverage_path),
        branch_path.name:sha256(branch_path),
        cases_path.name:sha256(cases_path),
    }
}
summary_path.write_text(json.dumps(summary,indent=2)+"\n")

print("===== CYCLE 25 MONDRIAN V2 EVALUATION COMPLETE =====")
print("rows:",len(y))
print("positives:",int((y==1).sum()))
print("primary policy: mondrian_v2_q90")
print("policy reselection performed: False")
print("supplementary 2026 outcomes read: False")
print("\nBranch metrics:")
print(branch_metrics.to_string(index=False))
print("\nMondrian class-conditional coverage:")
print(coverage.to_string(index=False))
print("\nOperational policy results:")
print(results.to_string(index=False))
print("\nPrimary q90 vs baseline:")
print(json.dumps(comparison,indent=2))
print("\nSaved:",summary_path)
print("Saved:",results_path)
print("Saved:",coverage_path)
print("Saved:",branch_path)
print("Saved:",cases_path)


===== CYCLE 25 MONDRIAN V2 EVALUATION COMPLETE =====
rows: 35846
positives: 3860
primary policy: mondrian_v2_q90
policy reselection performed: False
supplementary 2026 outcomes read: False

Branch metrics:
   branch  available_cases  availability_rate    brier  log_loss   tp    fp   fn    tn   recall  specificity  precision       f1      tss      hss
    sharp            35846                1.0 0.077365  0.267044 2811  5992 1049 25994 0.728238     0.812668   0.319323 0.443971 0.540906 0.346066
      aia            35846                1.0 0.097319  0.363898 2869 12911  991 19075 0.743264     0.596355   0.181812 0.292159 0.339619 0.144046
sharp_aia            35846                1.0 0.081192  0.273981 2615  6060 1245 25926 0.677461     0.810542   0.301441 0.417232 0.488003 0.315159

Mondrian class-conditional coverage:
   branch  alpha  available_cases  overall_coverage  class0_coverage  class1_coverage  empty_rate  singleton_no_flare_rate  singleton_flare_rate  doubleton_rate
    sha